# LeetCode #51: N-Queens

https://leetcode.com/problems/n-queens/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^n)$ | $O(n^2)$ |
| **Optimal: Backtracking with Bitmask ★** | $O(n!)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Place a queen in every cell of every row and check all $n^n$ combinations for validity. Most combinations violate column or diagonal constraints immediately, so this approach wastes enormous effort revisiting illegal states.

### Optimal: Backtracking with Bitmask ★
Recurse row by row. Track which columns and two diagonal families are already attacked using three integer bitmasks. At each row, only try columns where none of the three masks are set — invalid columns are pruned instantly without scanning the board. When all $n$ rows are filled, reconstruct the board string.

**Constraints:**
* 1 <= n <= 9

## Solutions
### C#

In [ ]:
// Backtracking with bitmask: track attacked columns and diagonals as bits
public class Solution {
    private List<IList<string>> results = new();
    private int n;
    private int[] queens; // queens[row] = col

    public IList<IList<string>> SolveNQueens(int n) {
        this.n = n;
        queens = new int[n];
        // Explore all placements starting from row 0 with no columns/diagonals attacked
        Backtrack(0, 0, 0, 0);
        return results;
    }

    private void Backtrack(int row, int colMask, int diagMask, int antiDiagMask) {
        if (row == n) {
            // All rows filled — reconstruct board strings
            results.Add(BuildBoard());
            return;
        }
        // Available columns: bits not set in any of the three masks
        int available = ((1 << n) - 1) & ~(colMask | diagMask | antiDiagMask);
        while (available != 0) {
            // Isolate the lowest set bit (rightmost free column)
            int bit = available & (-available);
            available -= bit;
            int col = BitOperations.Log2((uint)bit);
            queens[row] = col;
            // Shift diagonal masks one step: left diagonal shifts right, anti-diagonal shifts left
            Backtrack(row + 1,
                      colMask | bit,
                      (diagMask | bit) >> 1,
                      (antiDiagMask | bit) << 1);
        }
    }

    private List<string> BuildBoard() {
        var board = new List<string>();
        for (int r = 0; r < n; r++) {
            var row = new char[n];
            Array.Fill(row, '.');
            row[queens[r]] = 'Q';
            board.Add(new string(row));
        }
        return board;
    }
}

### Python

In [ ]:
# Backtracking with bitmask: track attacked columns and diagonals as bits
from typing import List

class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:
        results = []
        queens = [-1] * n  # queens[row] = column index

        def backtrack(row, col_mask, diag_mask, anti_mask):
            if row == n:
                # All rows filled — reconstruct board strings
                board = []
                for r in range(n):
                    board.append('.' * queens[r] + 'Q' + '.' * (n - queens[r] - 1))
                results.append(board)
                return
            # Available columns: bits not attacked by any mask
            available = ((1 << n) - 1) & ~(col_mask | diag_mask | anti_mask)
            while available:
                # Isolate lowest set bit (rightmost free column)
                bit = available & (-available)
                available -= bit
                col = bit.bit_length() - 1
                queens[row] = col
                # Propagate diagonal masks one step down the board
                backtrack(row + 1,
                          col_mask | bit,
                          (diag_mask | bit) >> 1,
                          (anti_mask | bit) << 1)

        backtrack(0, 0, 0, 0)
        return results

### Go

In [ ]:
// Backtracking with bitmask: track attacked columns and diagonals as bits
package main

import "math/bits"

func solveNQueens(n int) [][]string {
    results := [][]string{}
    queens := make([]int, n)
    full := (1 << n) - 1

    var backtrack func(row, colMask, diagMask, antiMask int)
    backtrack = func(row, colMask, diagMask, antiMask int) {
        if row == n {
            // All rows filled — reconstruct board strings
            board := make([]string, n)
            for r := 0; r < n; r++ {
                rowBytes := make([]byte, n)
                for c := range rowBytes {
                    rowBytes[c] = '.'
                }
                rowBytes[queens[r]] = 'Q'
                board[r] = string(rowBytes)
            }
            results = append(results, board)
            return
        }
        // Available columns: bits not attacked by any mask
        available := full & ^(colMask | diagMask | antiMask)
        for available != 0 {
            // Isolate lowest set bit (rightmost free column)
            bit := available & (-available)
            available -= bit
            col := bits.TrailingZeros(uint(bit))
            queens[row] = col
            // Shift diagonal masks as the queen's influence propagates down
            backtrack(row+1,
                colMask|bit,
                (diagMask|bit)>>1,
                (antiMask|bit)<<1)
        }
    }

    backtrack(0, 0, 0, 0)
    return results
}

### Rust

In [ ]:
// Backtracking with bitmask: track attacked columns and diagonals as bits
impl Solution {
    pub fn solve_n_queens(n: i32) -> Vec<Vec<String>> {
        let n = n as usize;
        let mut results = Vec::new();
        let mut queens = vec![0usize; n];
        let full = (1 << n) - 1usize;

        Self::backtrack(0, 0, 0, 0, n, full, &mut queens, &mut results);
        results
    }

    fn backtrack(
        row: usize, col_mask: usize, diag_mask: usize, anti_mask: usize,
        n: usize, full: usize, queens: &mut Vec<usize>, results: &mut Vec<Vec<String>>,
    ) {
        if row == n {
            // All rows filled — reconstruct board strings
            let board = queens.iter().map(|&col| {
                let mut row_str = vec![b'.'; n];
                row_str[col] = b'Q';
                String::from_utf8(row_str).unwrap()
            }).collect();
            results.push(board);
            return;
        }
        // Available columns: bits not attacked by any mask
        let mut available = full & !(col_mask | diag_mask | anti_mask);
        while available != 0 {
            // Isolate lowest set bit (rightmost free column)
            let bit = available & available.wrapping_neg();
            available -= bit;
            let col = bit.trailing_zeros() as usize;
            queens[row] = col;
            // Propagate diagonal masks one step toward the next row
            Self::backtrack(
                row + 1,
                col_mask | bit,
                (diag_mask | bit) >> 1,
                (anti_mask | bit) << 1,
                n, full, queens, results,
            );
        }
    }
}

## Example Scenarios

**1. Common Case** — $n = 4$

**Input:** `n = 4`
BFS explores row-by-row with bitmasks. Two valid solutions exist: `[".Q..","...Q","Q...","..Q."]` and `["..Q.","Q...","...Q",".Q.."]`. Both are found and returned.

**2. Slightly Complex** — $n = 6$

**Input:** `n = 6`
For $n = 6$ there are 4 distinct solutions. The bitmask pruning eliminates most of the $6^6 = 46{,}656$ brute-force states; only $O(6!)$ paths are explored. Each solution is a permutation of columns with no diagonal conflicts.

**3. Edge Case: Time Factor** — $n = 9$

**Input:** `n = 9`
Worst-case input. There are 352 solutions for $n = 9$. The bitmask ensures each row processes at most $n$ bits, keeping actual work proportional to the number of valid partial states rather than $9^9 \approx 387$ million.

**4. Edge Case: Space Factor** — $n = 1$

**Input:** `n = 1`
Single cell: the queen occupies `(0, 0)`. Only one board is returned: `["Q"]`. The `queens` array and recursion stack are both $O(1)$ in this degenerate case.

**5. Almost-Impossible but Plausible** — $n = 8$ (classic 8-Queens)

**Input:** `n = 8`
The classic chess puzzle. 92 distinct solutions exist. The bitmask method explores roughly $O(8!) = 40{,}320$ leaf paths versus $8^8 \approx 16.7$ million brute-force combinations, a $\approx 400\times$ speedup.